# 中断

## 在任意位置中断

LangGraph允许你在图执行期间任意代码位置执行中断。

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig
from langgraph_python.graphs.demo import (
    demo_interrupt_graph,
    demo_interrupt_parallel_graph,
    demo_subgraph_interrupt_graph,
)


checkpointer = InMemorySaver()
config:RunnableConfig = {
    "configurable":{
        "thread_id": "interrupt_demo"
    }
}
graph = demo_interrupt_parallel_graph.build_graph().compile(checkpointer)

## LangGraph API

### 运行到中断

In [ ]:
result = await graph.ainvoke(
    input={}, # type: ignore
    config=config
)
result

In [ ]:
history = list(graph.get_state_history(config=config))
history

### 多种方式查看中断

In [ ]:
interrupts = result.get("__interrupt__", [])
interrupts

In [ ]:
cur_cp = graph.get_state(config=config)
cp_interrupts = cur_cp.interrupts
cp_interrupts

### 中断的恢复

In [ ]:
# 构建回答
interrupts = result.get("__interrupt__", [])
answers = {
    inter.id: True
    for inter in interrupts
}
answers

In [ ]:
from langgraph.types import Command

result = await graph.ainvoke(input=Command(resume=answers), config=config)

result

## Agent Server API

In [ ]:
from langgraph_sdk import get_client
from langgraph_sdk.schema import Command as SDKCommand

# 连接本地 Agent Server
client = get_client(url="http://localhost:2024")

# 直接使用 langgraph.json 中注册的图名称作为 assistant_id
assistant_id = "interrupt_parallel_graph"

# 删除所有线程，免得看晕了
threads = await client.threads.search()
for t in threads:
    await client.threads.delete(thread_id=t['thread_id'])

# 创建新线程
thread = await client.threads.create(metadata={"__name__": "中断"})
thread_id = thread["thread_id"]
thread_id

### 运行到中断

In [ ]:
result = await client.runs.wait(
    thread_id=thread_id,
    assistant_id=assistant_id,
    input={},
)
result

### 多种方式查看中断

In [ ]:
interrupts = result.get("__interrupt__", []) # type: ignore

interrupts

In [ ]:
cur_cp = await client.threads.get_state(thread_id=thread_id)
cp_interrupts = cur_cp['interrupts']

cp_interrupts

### 中断恢复

In [ ]:
answers = {
    inter["id"]: True
    for inter in interrupts
}
answers

In [ ]:
# 同时恢复 A、B 的第一个中断，本次 run 会运行到 A、B 的第二个中断
result = await client.runs.wait(
    thread_id=thread_id,
    assistant_id=assistant_id,
    command=SDKCommand(resume=answers),
)
result

## 最佳实践

1. 永远不要捕获`interrupt`异常，至少不要捕获`GraphBubbleUp`异常，至至少少不要吞掉异常

2. 被中断的节点，在恢复时会重新运行，因此保证中断前副作用的幂等性

3. 节点运行过程中，如果要依次引发多个中断，必须确保中断的数量和顺序是稳定的

4. 子图的`checkpoint`必须设置为`None`或`True`才能正常的在父图中中断

5. 时间旅行会重新中断